# Stage-gate playground — tilearray 0.2

Drive the library like a data scientist before cutting **0.2.0**: pick a layer, set a point + box width, fetch, plot, nudge the AOI, try again.

```bash
uv sync --dev
uv run jupyter lab   # open this notebook
```

Habit: `config → create_array → compute_with_policy → plot`.


## Setup


In [ ]:
from __future__ import annotations

import time

import matplotlib.pyplot as plt
import numpy as np

import tilearray
from tilearray import WCSConfig, WMSConfig, XYZConfig, compute_with_policy, create_array
from tilearray.types import CRS, Format

print("tilearray", tilearray.__version__)


def box_around(x: float, y: float, width: float) -> tuple[float, float, float, float]:
    """BBox centred on (x, y); width is the full edge length in x/y units."""
    half = width / 2.0
    return (x - half, y - half, x + half, y + half)


def summarize(arr, *, wall_s: float | None = None) -> None:
    if wall_s is not None:
        print(f"wall_s={wall_s:.2f}")
    print(f"shape={arr.shape}  dtype={arr.dtype}  dims={arr.dims}")
    for key in ("crs", "service_type", "nodata", "band_policy"):
        if key in arr.attrs:
            print(f"  {key}={arr.attrs[key]!r}")
    fp = arr.attrs.get("tilearray_fetch_policy") or {}
    if fp:
        keep = ("max_concurrent", "call_max_concurrent", "adaptive_concurrency")
        print("  fetch:", {k: fp[k] for k in keep if k in fp})
    te = arr.attrs.get("tilearray_tile_errors")
    if te is not None:
        keep = ("armed", "soft_failed", "aborted", "policy")
        print("  tile_errors:", {k: te[k] for k in keep if k in te})


def show(arr, *, title: str = "", cmap: str | None = None) -> None:
    plt.figure(figsize=(6, 5))
    if "band" in getattr(arr, "dims", ()) and arr.sizes.get("band", 0) >= 3:
        rgb = np.asarray(arr.transpose("y", "x", "band")[..., :3])
        if np.issubdtype(rgb.dtype, np.floating):
            scale = np.nanmax(rgb) or 1.0
            rgb = np.clip(rgb / scale, 0, 1)
        plt.imshow(rgb)
    else:
        data = np.asarray(arr).squeeze()
        use_cmap = cmap or "gray"
        plt.imshow(data, cmap=use_cmap)
        if cmap is not None:
            plt.colorbar(shrink=0.8)
    if title:
        plt.title(title)
    plt.axis("off")
    plt.tight_layout()
    plt.show()


## Layers

| `LAYER` | What | CRS / width units |
|---------|------|-------------------|
| `osm` | OpenStreetMap XYZ (`for_openstreetmap`) | lon/lat degrees |
| `ea_lidar` | EA Lidar DTM 1 m WCS (`for_ea_dsp`) | OSGB metres |
| `gibs` | NASA GIBS Blue Marble XYZ | lon/lat degrees |
| `ea_vom` | EA Vegetation Object Model WMS | OSGB metres |

Sample centres (also commented in the config cell): London / Leeds / Manchester for OSM; `E,N = 432300, 383100` for EA.


## Config — edit these


In [ ]:
# --- pick layer + AOI (edit me) -------------------------------------------
LAYER = "osm"  # "osm" | "ea_lidar" | "gibs" | "ea_vom"

# OSM / GIBS: lon/lat + width in degrees (~0.01° ≈ 1 km)
LON, LAT = -0.1276, 51.5072   # London
# LON, LAT = -1.5491, 53.8008  # Leeds
# LON, LAT = -2.2426, 53.4808  # Manchester
WIDTH_DEG = 0.01

# EA Lidar / VOM: OSGB metres
E, N = 432_300.0, 383_100.0
# E, N = 430_500.0, 380_500.0
WIDTH_M = 800.0

OSM_ZOOM = 15
GIBS_ZOOM = 6
EA_CHUNK = (128, 128)
EA_VOM_LAYER = "Vegetation_Object_Model_2022"  # confirm year if 404

MAX_CONCURRENT = None  # e.g. 1 (OSM) or 4 (EA); None = preset default


## Run


In [ ]:
extra = {}
if MAX_CONCURRENT is not None:
    extra["max_concurrent_requests"] = MAX_CONCURRENT

if LAYER == "osm":
    x, y, width = LON, LAT, WIDTH_DEG
    bbox = box_around(x, y, width)
    crs = CRS.EPSG_4326
    config = XYZConfig.for_openstreetmap(zoom=OSM_ZOOM, output_format=Format.PNG, **extra)
    title = "OSM XYZ  © OpenStreetMap contributors"

elif LAYER == "ea_lidar":
    x, y, width = E, N, WIDTH_M
    bbox = box_around(x, y, width)
    crs = CRS.EPSG_27700
    config = WCSConfig.for_ea_dsp(
        "https://environment.data.gov.uk/spatialdata/"
        "lidar-composite-digital-terrain-model-dtm-1m/wcs",
        coverage_id="13787b9a-26a4-4775-8523-806d13af58fc__Lidar_Composite_Elevation_DTM_1m",
        crs=CRS.EPSG_27700,
        output_format=Format.GEOTIFF,
        chunk_size=EA_CHUNK,
        resolution=(1.0, 1.0),
        **extra,
    )
    title = "EA Lidar DTM"

elif LAYER == "gibs":
    x, y, width = LON, LAT, WIDTH_DEG
    bbox = box_around(x, y, width)
    crs = CRS.EPSG_4326
    config = XYZConfig.from_url(
        "https://gibs.earthdata.nasa.gov/wmts/epsg3857/best/"
        "BlueMarble_NextGeneration/default/GoogleMapsCompatible_Level8/{z}/{y}/{x}.jpeg",
        zoom=GIBS_ZOOM,
        output_format=Format.JPEG,
        headers={"User-Agent": f"tilearray/{tilearray.__version__}"},
        **extra,
    )
    title = "GIBS Blue Marble"

elif LAYER == "ea_vom":
    x, y, width = E, N, WIDTH_M
    bbox = box_around(x, y, width)
    crs = CRS.EPSG_27700
    config = WMSConfig.from_url(
        "https://environment.data.gov.uk/spatialdata/vegetation-object-model/wms",
        layers=EA_VOM_LAYER,
        crs=CRS.EPSG_27700,
        output_format=Format.PNG,
        chunk_size=EA_CHUNK,
        **extra,
    )
    title = f"EA VOM ({EA_VOM_LAYER})"

else:
    raise ValueError(f"Unknown LAYER={LAYER!r}")

print(f"{LAYER}: centre=({x}, {y}) width={width} bbox={bbox}")

da = create_array(config, bbox, crs)
t0 = time.perf_counter()
arr = compute_with_policy(da)
summarize(arr, wall_s=time.perf_counter() - t0)
show(arr, title=title, cmap="terrain" if LAYER == "ea_lidar" else None)


## Scratch — paste & tweak

Same pattern without the layer switch. Change the numbers and re-run.


In [ ]:
# Small EA Lidar window — change E/N/WIDTH and re-run
E, N = 432_300.0, 383_100.0
WIDTH = 400.0  # metres

bbox = box_around(E, N, WIDTH)
config = WCSConfig.for_ea_dsp(
    "https://environment.data.gov.uk/spatialdata/"
    "lidar-composite-digital-terrain-model-dtm-1m/wcs",
    coverage_id="13787b9a-26a4-4775-8523-806d13af58fc__Lidar_Composite_Elevation_DTM_1m",
    crs=CRS.EPSG_27700,
    output_format=Format.GEOTIFF,
    chunk_size=(64, 64),
    resolution=(1.0, 1.0),
)

da = create_array(config, bbox, CRS.EPSG_27700)
t0 = time.perf_counter()
arr = compute_with_policy(da)
summarize(arr, wall_s=time.perf_counter() - t0)
show(arr, title="EA Lidar", cmap="terrain")


In [ ]:
# OSM around a lon/lat — change LON/LAT/WIDTH/ZOOM and re-run
LON, LAT = -0.1276, 51.5072
WIDTH = 0.008  # degrees
ZOOM = 16

bbox = box_around(LON, LAT, WIDTH)
config = XYZConfig.for_openstreetmap(zoom=ZOOM)

da = create_array(config, bbox, CRS.EPSG_4326)
t0 = time.perf_counter()
arr = compute_with_policy(da)
summarize(arr, wall_s=time.perf_counter() - t0)
show(arr, title="OSM  © OpenStreetMap contributors")


## Optional — soft-fail peek

Default mosaics hard-fail after retries. This uses a deliberate 404 URL so you can see `tilearray_tile_errors`.


In [ ]:
# Deliberate 404 mosaic — inspect soft-fail attrs (GeoTIFF skips band probe)
bbox = box_around(0.0, 0.0, 0.01)
config = XYZConfig.from_url(
    "https://httpbingo.org/status/404",
    zoom=1,
    output_format=Format.GEOTIFF,
    tile_size=64,
    fetch_retries=0,
)

da = create_array(config, bbox, CRS.EPSG_4326, soft_fail=True, soft_fail_max_fraction=1.0)
arr = compute_with_policy(da)
summarize(arr)
print("finite fraction:", float(np.isfinite(arr.values).mean()))


## Notes

- Start small (`WIDTH` low). Bigger boxes → more tiles → slower / more host pressure.
- OSM preset ceiling is 2; EA AIMD ceiling is 10 — don't raise past those for stage-gate.
- `load_array` is eager sugar; prefer lazy + `compute_with_policy`.
- Private/keyed hosts: `params=` / `headers=` on the config (env-backed secrets).

### What felt wrong?

- [ ] Happy path clear?
- [ ] Moving AOI / swapping layer easy enough?
- [ ] Timing / attrs useful?
- [ ] Anything blocking a **0.2.0** cut?

```
(your notes)
```
